# 9 WorkFlow Analista Jr

### Consideraciones:

1. Este cuaderno está modificado y adaptado para incorporar las mejores alternativas en el logro de la optimización del modelo LightGBM.

2. En todos los casos, según se aclarará punto por punto, los criterios adoptados se basarán en las recomendaciones realizadas por mis compañeros en sus respectivos experimentos.

### 9.1 Objetivo

### Optimizar el modelo de predicción basado en LightGBM empleando los resultados y correspondientes recomendaciones de los experimentos encarados por los grupos del curso.



## 9.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

#### Parametros

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 402372  # Corresponde a mi semilla primigenia personal

#************* Aplico Algoritmo Genético ****************************************
# Configuración del Algoritmo Genético de Feature Engineering Intra-mes (gramEvol)
PARAM$GA <- list(
  popSize = 150,            # Tamaño de la población de individuos
  iterations = 50,          # Cantidad de generaciones evolutivas
  top_features = 20,        # Cantidad de mejores features no lineales a inyectar al dataset
  max_terminales = 60,      # Forzar a usar todas las variables
  seqLen = 250,             # Longitud máxima de codones del genoma
  max.depth = 10,           # Profundidad máxima del árbol sintáctico BNF
  max_filas_fitness = 50000 # Muestra máxima para evaluación de fitness ultrarrápida
)
#********************************************************************************

PARAM$experimento <- 100005          # Identifico este experimento como el último gran experimento
PARAM$dataset <- "analistajr_competencia_2026.csv.gz"

#### Carpeta del Experimento

In [ ]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

### 9.3.1   Preprocesamiento del dataset

#### 9.3.1.1  DT incorporar dataset

In [ ]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

#### 9.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"










<font color="blue">En función de los resultados obtenidos por los grupos que experimentaron distintos modos de tratar con variables faltantes, considero aceptar la recomendación de mantener el método propuesto por la cátedra.</font>

<font color="blue">Utilizaré el método **Machine Learning** que consiste en asignar NA a los valores faltantes.</font>

In [ ]:
AsignarNA_campomeses <- function(pcampo, pmeses) {

  if( pcampo %in% colnames( dataset ) ) {

    dataset[ foto_mes %in% pmeses, paste0(pcampo) := NA ]
  }
}

In [ ]:

Corregir_atributo <- function(pcampo, pmeses, pmetodo)
{
  # si el campo no existe en el dataset, Afuera !
  if( !(pcampo %in% colnames( dataset )) )
    return( 1 )

  # llamo a la funcion especializada que corresponde
  switch( pmetodo,
    "MachineLearning"     = AsignarNA_campomeses(pcampo, pmeses),
  )

  return( 0 )
}

In [ ]:

Corregir_Rotas <- function(dataset, pmetodo) {
  gc(verbose= FALSE)
  cat( "inicio Corregir_Rotas()\n")
  # acomodo los errores del dataset

  Corregir_atributo("active_quarter", c(202006), pmetodo) # 1
  Corregir_atributo("internet", c(202006), pmetodo) # 2

  Corregir_atributo("mrentabilidad", c(201905, 201910, 202006), pmetodo) # 3
  Corregir_atributo("mrentabilidad_annual", c(201905, 201910, 202006), pmetodo) # 4

  Corregir_atributo("mcomisiones", c(201905, 201910, 202006), pmetodo) # 5

  Corregir_atributo("mactivos_margen", c(201905, 201910, 202006), pmetodo) # 6
  Corregir_atributo("mpasivos_margen", c(201905, 201910, 202006), pmetodo) # 7

  Corregir_atributo("mcuentas_saldo", c(202006), pmetodo) # 8

  Corregir_atributo("ctarjeta_debito_transacciones", c(202006), pmetodo) # 9

  Corregir_atributo("mautoservicio", c(202006), pmetodo) # 10

  Corregir_atributo("ctarjeta_visa_transacciones", c(202006), pmetodo) # 11
  Corregir_atributo("mtarjeta_visa_consumo", c(202006), pmetodo) # 12

  Corregir_atributo("ctarjeta_master_transacciones", c(202006), pmetodo) # 13
  Corregir_atributo("mtarjeta_master_consumo", c(202006), pmetodo) # 14

  Corregir_atributo("ctarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 15
  Corregir_atributo("mttarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 16

  Corregir_atributo("ccajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 17

  Corregir_atributo("mcajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 18

  Corregir_atributo("ctarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 19

  Corregir_atributo("mtarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 20

  Corregir_atributo("ctarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 21

  Corregir_atributo("mtarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 22

  Corregir_atributo("ccomisiones_otras", c(201905, 201910, 202006), pmetodo) # 23
  Corregir_atributo("mcomisiones_otras", c(201905, 201910, 202006), pmetodo) # 24

  Corregir_atributo("cextraccion_autoservicio", c(202006), pmetodo) # 25
  Corregir_atributo("mextraccion_autoservicio", c(202006), pmetodo) # 26

  Corregir_atributo("ccheques_depositados", c(202006), pmetodo) # 27
  Corregir_atributo("mcheques_depositados", c(202006), pmetodo) # 28
  Corregir_atributo("ccheques_emitidos", c(202006), pmetodo) # 29
  Corregir_atributo("mcheques_emitidos", c(202006), pmetodo) # 30
  Corregir_atributo("ccheques_depositados_rechazados", c(202006), pmetodo) # 31
  Corregir_atributo("mcheques_depositados_rechazados", c(202006), pmetodo) # 32
  Corregir_atributo("ccheques_emitidos_rechazados", c(202006), pmetodo) # 33
  Corregir_atributo("mcheques_emitidos_rechazados", c(202006), pmetodo) # 34

  Corregir_atributo("tcallcenter", c(202006), pmetodo) # 35
  Corregir_atributo("ccallcenter_transacciones", c(202006), pmetodo) # 36

  Corregir_atributo("thomebanking", c(202006), pmetodo) # 37
  Corregir_atributo("chomebanking_transacciones", c(201910, 202006), pmetodo) # 38

  Corregir_atributo("ccajas_transacciones", c(202006), pmetodo) # 39
  Corregir_atributo("ccajas_consultas", c(202006), pmetodo) # 40

  Corregir_atributo("ccajas_depositos", c(202006, 202105), pmetodo) # 41

  Corregir_atributo("ccajas_extracciones", c(202006), pmetodo) # 41
  Corregir_atributo("ccajas_otras", c(202006), pmetodo) # 43

  Corregir_atributo("catm_trx", c(202006), pmetodo) # 44
  Corregir_atributo("matm", c(202006), pmetodo) # 45
  Corregir_atributo("catm_trx_other", c(202006), pmetodo) # 46
  Corregir_atributo("matm_other", c(202006), pmetodo) # 47

  cat( "fin Corregir_rotas()\n")
}


In [ ]:
# resuelvo el Catastrophe Analysis

setorder( dataset, numero_de_cliente, foto_mes )

PARAM$CA$metodo= "MachineLearning"

if( PARAM$CA$metodo %in% c("MachineLearning") )
  Corregir_Rotas(dataset, PARAM$CA$metodo)

#### 9.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, ajustando por algunos indices financieros

<font color="blue">Surge acá una situación a la hora de tomar decisión por una metodología de corrección.

<font color="blue">El grupo A determinó como metodología superadora aplicar Estandarización. El grupo B, sin confirmación estadística por ser más meticuloso a la hora de evaluar, recomienda Rank_cero_fijo.

<font color="blue">En función del trabajo realizado por el grupo A y los resultados obtenidos, decido aplicar Estandarización (aún cuando detecto un error en la fórmula que termina poniendo NA en todos los valores de los campos afectados).

## **Segunda opción**

<font color="blue">No teniendo evidencia sobre esta apartado de Data Drifting, decido no hacer ninguna corrección. Esta alternativa dio también buen resultado en el grupo A, y tiene validez estadística.

In [ ]:
# meses que me interesan para el ajuste de variables monetarias
vfoto_mes <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107, 202108, 202109
)

In [ ]:
drift_estandarizar <- function(campos_drift) {

  cat( "inicio drift_estandarizar()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_normal") :=
      (get(campo) -mean(campo, na.rm=TRUE)) / sd(get(campo), na.rm=TRUE),
      by = list(foto_mes)]

    dataset[, (campo) := NULL]
  }
  cat( "fin drift_estandarizar()\n")
}


In [ ]:
# por como armé los nombres de campos,
#  estos son los campos que expresan variables monetarias
campos_monetarios <- colnames(dataset)
campos_monetarios <- campos_monetarios[campos_monetarios %like%
  "^(m|Visa_m|Master_m|vm_m)"]

campos_monetarios

In [ ]:
# ejecuto el Data Drifting
setorder( dataset, numero_de_cliente, foto_mes )


PARAM$DR$metodo <- "estandarizar"

switch(PARAM$DR$metodo,
  "ninguno"        = cat("No hay correccion del data drifting"),
  "estandarizar"   = drift_estandarizar(campos_monetarios)
)


In [ ]:
colnames(dataset)

#### 9.3.1.3.1  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [ ]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]


In [ ]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

#### 9.3.1.3.2 FE_intra_GA: Feature Engineering Intra-mes mediante Algoritmo Genético (gramEvol)

<font color="blue">Implemento el Algoritmo Genético conforme la recomendación del grupo A que exploró este método de optimización.

In [ ]:
# ==============================================================================
# 9.3.1.3.2 FE_intra_GA: Algoritmo Genético (Grammatical Evolution)
# ==============================================================================

if (!require("gramEvol")) {
  install.packages("gramEvol", repos = "https://cloud.r-project.org", dependencies = TRUE)
}
if (!require("gramEvol")) {
  install.packages("gramEvol", repos = "https://cran.rstudio.com", dependencies = TRUE)
}
require("gramEvol")
require("lightgbm")
require("data.table")
require("parallel")

if (is.null(PARAM$GA)) {
  PARAM$GA <- list(
    popSize = 150,
    iterations = 50,
    top_features = 5,
    max_terminales = 60,
    seqLen = 250,
    max.depth = 10,
    max_filas_fitness = 50000
  )
}

# 1. Variables candidatas para el Algoritmo Genético
excluir_GA <- c("numero_de_cliente", "foto_mes", "clase_ternaria", "clase01", "azar", "fold_train", "fold_final_train")
candidatas_GA <- setdiff(colnames(dataset), excluir_GA)

# Filtrar únicamente variables numéricas
son_numericas <- sapply(dataset[, candidatas_GA, with = FALSE], is.numeric)
candidatas_GA <- candidatas_GA[son_numericas]

# Excluir fechas
patron_fechas <- "^(f|.*_f|.*fecha)"
candidatas_GA <- candidatas_GA[!grepl(patron_fechas, candidatas_GA, ignore.case = TRUE)]

# Límite de seguridad de terminales para controlar el espacio de búsqueda
MAX_TERMINALES <- if (!is.null(PARAM$GA$max_terminales)) PARAM$GA$max_terminales else 60
if (length(candidatas_GA) > MAX_TERMINALES) {
  set.seed(PARAM$semilla_primigenia)
  candidatas_GA <- sample(candidatas_GA, MAX_TERMINALES)
}

cat("Variables candidatas para Grammatical Evolution:", length(candidatas_GA), "\n")

# 2. Split Local de Validación (sin tocar 202107 ni 202109)
meses_disponibles <- sort(unique(dataset$foto_mes[dataset$foto_mes < 202107]))
meses_val_GA <- tail(meses_disponibles, 3)
meses_tr_GA  <- setdiff(meses_disponibles, meses_val_GA)

idx_tr_all <- which(dataset$foto_mes %in% meses_tr_GA)
idx_val_all <- which(dataset$foto_mes %in% meses_val_GA)

# Subsampling controlado para evaluación de fitness ultrarrápida
set.seed(PARAM$semilla_primigenia)
n_fit <- if (!is.null(PARAM$GA$max_filas_fitness)) PARAM$GA$max_filas_fitness else 50000
idx_tr_GA <- if (length(idx_tr_all) > n_fit) sample(idx_tr_all, n_fit) else idx_tr_all
idx_val_GA <- if (length(idx_val_all) > (n_fit / 2)) sample(idx_val_all, n_fit / 2) else idx_val_all

y_tr_GA <- ifelse(dataset$clase_ternaria[idx_tr_GA] %in% c("BAJA+1", "BAJA+2"), 1L, 0L)
y_val_GA <- ifelse(dataset$clase_ternaria[idx_val_GA] %in% c("BAJA+1", "BAJA+2"), 1L, 0L)

# 3. Operadores Protegidos
protected_div <- function(x, y) {
  res <- x / (y + 1e-5)
  res[is.na(res) | is.infinite(res)] <- 0
  res
}

protected_log_diff <- function(x, y) {
  res <- log(abs(x - y) + 1)
  res[is.na(res) | is.infinite(res)] <- 0
  res
}

es_expresion_trivial <- function(f) {
  !grepl("[+*/-]|protected_", f)
}

# 4. Definición de la Gramática BNF
string_vars <- paste(candidatas_GA, collapse = " | ")
rule_text <- paste0(
  "<expr> ::= <op>\n",
  "<op>   ::= <op> + <op> | <op> - <op> | <op> * <op> | ",
  "protected_div(<op>, <op>) | protected_log_diff(<op>, <op>) | <var>\n",
  "<var>  ::= ", string_vars
)

tf <- tempfile()
writeLines(rule_text, tf)
bnf_grammar <- CreateGrammar(tf)
unlink(tf)

# 5. Función de Fitness con LightGBM Univariado Real
fitness_gramEvol <- function(expr) {
  valores <- tryCatch(eval(expr, envir = dataset), error = function(e) NULL)
  if (is.null(valores)) return(1)

  val_tr <- valores[idx_tr_GA]
  val_finitos <- val_tr[is.finite(val_tr)]
  if (length(val_finitos) == 0 || length(unique(val_finitos)) <= 1) {
    return(1)
  }

  dtr_ga  <- lgb.Dataset(data = matrix(val_tr, ncol = 1), label = y_tr_GA, free_raw_data = TRUE)
  dval_ga <- lgb.Dataset(data = matrix(valores[idx_val_GA], ncol = 1), label = y_val_GA, free_raw_data = TRUE)

  modelo_ga <- tryCatch({
    lgb.train(
      params = list(objective = "binary", metric = "auc",
                    learning_rate = 0.1, num_threads = 1, verbosity = -1),
      data = dtr_ga, valids = list(valid = dval_ga),
      nrounds = 50, early_stopping_rounds = 10, verbose = -1
    )
  }, error = function(e) NULL)

  if (is.null(modelo_ga) || is.null(modelo_ga$best_score) || is.na(modelo_ga$best_score)) return(1)

  1 - modelo_ga$best_score
}

evaluar_genoma <- function(genoma) {
  expr_obj <- tryCatch(suppressWarnings(GrammarMap(genoma, bnf_grammar)), error = function(e) NULL)
  if (is.null(expr_obj) || !isTRUE(GrammarIsTerminal(expr_obj))) {
    return(list(score = 0, formula = NA_character_))
  }
  expr_lang <- tryCatch(as.expression(expr_obj), error = function(e) NULL)
  if (is.null(expr_lang) || length(expr_lang) == 0) {
    return(list(score = 0, formula = NA_character_))
  }

  expr_final  <- expr_lang[[1]]
  formula_str <- paste(deparse(expr_final, width.cutoff = 500L), collapse = " ")
  costo <- fitness_gramEvol(expr_final)
  auc   <- 1 - costo

  list(score = auc, formula = formula_str)
}

# 6. Ejecución del Algoritmo Genético
set.seed(PARAM$semilla_primigenia)

cat("\n===================================================================\n")
cat(">>> INICIANDO GRAMMATICAL EVOLUTION (gramEvol) <<<\n")
cat("===================================================================\n")

ge_res <- GrammaticalEvolution(
  grammarDef      = bnf_grammar,
  evalFunc        = fitness_gramEvol,
  popSize         = PARAM$GA$popSize,
  iterations      = PARAM$GA$iterations,
  terminationCost = 0.10,
  seqLen          = PARAM$GA$seqLen,
  max.depth       = PARAM$GA$max.depth,

  # Nuevos parámetros de presión selectiva
  elitism         = as.integer(PARAM$GA$popSize * 0.50),
  mutationChance  = 0.15,

  monitorFunc     = function(result) {
    cat(sprintf("Gen %2d | Mejor Costo: %.5f (AUC: %.5f)\n",
                result$population$currentIteration,
                result$best$cost,
                1 - result$best$cost))
  }
)

# 7. Extracción e Inyección del Top N al Dataset
cat("\n=== Evaluando población final para extraer el Top", PARAM$GA$top_features, "===\n")

pop_matrix      <- ge_res$population$population
poblacion_final <- split(pop_matrix, row(pop_matrix))

n_cores <- max(1, detectCores() - 1)
resultados <- if (.Platform$OS.type == "unix") {
  mclapply(poblacion_final, evaluar_genoma, mc.cores = n_cores)
} else {
  lapply(poblacion_final, evaluar_genoma)
}

scores_finales   <- sapply(resultados, function(r) r$score)
formulas_finales <- sapply(resultados, function(r) r$formula)

ordenados <- order(scores_finales, decreasing = TRUE)

formulas_vistas <- character()
ga_cols_creadas <- character()
top_guardados   <- 0
idx             <- 1

dt_trazabilidad <- data.table(Variable=character(), AUC=numeric(), Formula=character())

while (top_guardados < PARAM$GA$top_features && idx <= length(ordenados)) {
  i <- ordenados[idx]
  idx <- idx + 1

  if (is.na(scores_finales[i]) || scores_finales[i] <= 0.50) next
  if (is.na(formulas_finales[i]) || formulas_finales[i] %in% formulas_vistas) next
  if (es_expresion_trivial(formulas_finales[i])) next

  eval_res <- tryCatch(
    eval(parse(text = formulas_finales[i])[[1]], envir = dataset),
    error = function(e) NULL
  )
  if (is.null(eval_res) || length(unique(eval_res[is.finite(eval_res)])) <= 1) next

  top_guardados <- top_guardados + 1
  formulas_vistas <- c(formulas_vistas, formulas_finales[i])

  nombre_col <- paste0("GA_Feature_", top_guardados)
  ga_cols_creadas <- c(ga_cols_creadas, nombre_col)

  cat(sprintf("[%s] AUC Univariado: %.5f | Fórmula: %s\n", nombre_col, scores_finales[i], formulas_finales[i]))
  dataset[, (nombre_col) := eval_res]
  dt_trazabilidad <- rbind(dt_trazabilidad, list(nombre_col, scores_finales[i], formulas_finales[i]))
}

fwrite(dt_trazabilidad,
       file = paste0("GA_trazabilidad_formulas_s",PARAM$semilla_primigenia, ".csv"), sep = ",")
cat("\nArchivo de trazabilidad guardado en: GA_trazabilidad_formulas_s", PARAM$semilla_primigenia, ".csv\n")

cat("\nColumnas generadas por Algoritmo Genético e inyectadas al dataset:", paste(ga_cols_creadas, collapse = ", "), "\n")

# Blindaje anti-leakage: asegura que clase01 no quede en dataset antes de FEhist
if ("clase01" %in% colnames(dataset)) dataset[, clase01 := NULL]


In [ ]:
# Auditoría de resultados del Algoritmo Genético e inspección de columnas
cat("Total individuos en población final:", length(scores_finales), "\n")
cat("Individuos con score > 0.50:", sum(scores_finales > 0.50, na.rm = TRUE), "\n")
cat("Expresiones no triviales con score > 0.50:", sum(scores_finales > 0.50 & !sapply(formulas_finales, es_expresion_trivial), na.rm = TRUE), "\n")
cat("Expresiones únicas no triviales:", length(unique(formulas_finales[scores_finales > 0.50 & !sapply(formulas_finales, es_expresion_trivial)])), "\n")
cat("\nColumnas actuales del dataset tras Feature Engineering Intra-mes (Manual + GA):\n")
colnames(dataset)


#### 9.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

In [ ]:
# No se implementa Feature Engineering a partir de Random Forest

#### 9.3.1.5  FEhist Feature Engineering historico

El Fature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

<font color="blue">Conforme recomendación del grupo que exploró la implementación de Feature Engineering Histórico, emplearé Lags de orden 3, Delta y medias móviles.

In [ ]:
# Feature Engineering Historico

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

#******* Implementación *****************************************
# lags de orden 3
dataset[,
    paste0(cols_lagueables, "_lag3") := shift(.SD, 3, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]
#****************************************************************


# agrego los delta lags *****3º Linea es nueva implementacion****************
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
    dataset[, paste0(vcol, "_delta3") := get(vcol) - get(paste0(vcol, "_lag3"))]
}

#******** Implementación ****************************************
# MEDIAS MOVILES (frollmean) - ventana de 3 meses
# aLign = "right": toma el mes actual y N-1 atrás
dataset[, paste0(cols_lagueables, "_ma3") :=
  frollmean(.SD, n = 3, align = "right",
  fill = NA, na.rm = TRUE),
  by = numero_de_cliente,
  .SDcols = cols_lagueables]
#****************************************************************





Verificacion de los campos recien creados

In [ ]:
ncol(dataset)
colnames(dataset)

#### 9.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  ni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [ ]:
# No se implementa la reduccion de la dimensionalidad con canaritos

<font color="blue">Tampoco resulta beneficioso realizar una reducción de dimensionalidad por PCA en función de los resultados obtenidos por el grupo que analizó esta temática.

<font color="blue"> (Coincide con mi sesgo personal, por el que considero que no puede haber beneficio reduciendo el universo de datos en esta metodología).

### 9.3.2 Modelado

#### 9.3.2.1 Training Strategy

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202109  obviamente completo

* final_train =  [ 201901, 202107 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202107   completo, sin undersampling
   * training = [ 201901, 202105 ]  donde se consideran el 100% de los CONTINUA

<font color="blue">En función de las conclusiones a las que llegamos en mi grupo particular, al haber explorado distintos escenarios con quita de meses afectados por la pandemia, considero adecuado no reducir el dataset realizando ningun tipo de quita.

<font color="blue">Respecto del undersampling, considero un valor del 10% conforme recomendó el grupo A al explorar la incidencia de este hiperparámetro.

In [ ]:
PARAM$trainingstrategy$validate <- c(202107)

PARAM$trainingstrategy$training <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105
)

#******* Implemento undersampling 10% ********
PARAM$trainingstrategy$training_pct <- 0.1


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [ ]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [ ]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar"))
)

In [ ]:
# preparo para que se puede hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]


if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  free_raw_data= TRUE
)

In [ ]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)

####  9.3.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * feature_fraction = 0.5  para poner algo que generalmente no falla
  * learning_rate = 0.03  para que aprenda lento


* Parametros que se optimizan en el Grid Search
  * num_leaves  [64, 512]
  * min_data_in_leaf  [64, 2048]

In [ ]:
# parametros fijos del LightGBM
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  seed= PARAM$semilla_primigenia,
  max_bin= 31,
  learning_rate= 0.03,
  feature_fraction= 0.5,
  num_iterations= 2048,  # valor grande, lo limita early_stopping_rounds
  early_stopping_rounds= 200,
  num_leaves= 64,
  min_data_in_leaf= 128
)


In [ ]:
# En  x llegan los parametros moviles de LightGBM
#  devuelve la AUC en validate del modelo entrenado
#  en el parametro x llegan los hiperparámetros que se estan optimizando

Estimar_AUC_lightgbm <- function(x) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " niter ", modelo_train$best_iter,
    " AUC ", AUC
  )

  # hago espacio en la memoria
  niter <- modelo_train$best_iter
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return( list(AUC, niter))
}

seteo del Grid Search

In [ ]:
# lo que sigue a continuacion es una forma alternativa a los loops anidados
# creo una tabla con el producto cartesiano de los vectores
tb_nueva <- CJ(
  num_leaves= c(64, 128, 256, 384, 512),
  min_data_in_leaf= c(64, 256, 512, 1024, 2048),
  feature_fraction= c(0.5, 0.8)
)

##### Corrida del Grid Search,  aqui se hace el trabajo pesado
<br> por favor no se asuste con los warnings que pudieran aparecer
<br> ATENCION, la siguiente celda demora 65 minutos
<br> una Analista Jr  debe ser capaz de tolerar estoicamente esta tortura
<br> (y masticar chicle al mismo tiempo)

In [ ]:
# registro a registro calculo la AUC
tb_nueva[,  c("AUC", "num_iterations"):= Estimar_AUC_lightgbm( .SD ),
  by=1:nrow(tb_nueva) ]

la optimizacion de hiperparámetros de tipo  Grid Search ha corrido, extraigo los mejores hiperparametros

In [ ]:
tb_nueva

fwrite( tb_nueva,
  file= "tb_grid_search_01.txt",
  sep="\t",
  append= TRUE
)

In [ ]:
setorder( tb_nueva, -AUC)  # ordeno DESCENDENTE por AUC
PARAM$out$lgbm$AUC <- tb_nueva[1, AUC] # en la posicion 1 estan los mejores
PARAM$out$lgbm$mejores_hiperparametros <- as.list( tb_nueva[1] )
PARAM$out$lgbm$mejores_hiperparametros$AUC <- NULL
PARAM$out$lgbm$mejores_hiperparametros

In [ ]:
tb_nueva

### 9.3.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización

In [ ]:
PARAM$trainingstrategy$final_train <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107
)


dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# creo el dfinal_train en formato  LightGBM
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= TRUE
)

nrow( dfinal_train) # verifico el tamaño

##### Final Training Hyperparameters

In [ ]:
# uno los parametros fijos y los mejores encontrados de los variables
fijos <- copy(PARAM$lgbm$param_fijos)

# quito lo que optimice en la Bayesian Optimization
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL

# agrego a los hiperparametros fijos los que encontre con la Bayesian Optimization
param_final <- c(fijos, PARAM$out$lgbm$mejores_hiperparametros)

##### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [ ]:
final_model <- lgb.train(
  data= dfinal_train,
  param= param_final,
  verbose= -100
)

In [ ]:
# grabo a disco el modelo en un formato para seres humanos ... ponele ...

lgb.save(final_model, "modelo.txt")

In [ ]:
# ahora imprimo la importancia de variables

tb_importancia <- as.data.table(lgb.importance(final_model))
archivo_importancia <- "impo.txt"

fwrite( tb_importancia,
  file= archivo_importancia,
  sep= "\t"
)

#### Scoring

Aplico el modelo final a los datos del futuro

In [ ]:
PARAM$trainingstrategy$future <- c(202109)

dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]

In [ ]:
# aplico final_model   a dfuture

prediccion <- predict(
  final_model,
  data.matrix(dfuture[, campos_buenos, with= FALSE])
)

##### Tabla Prediccion

In [ ]:
tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := prediccion]

# grabo las probabilidad del modelo
#  me va a ser util para hacer Ensembles de modelos
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

#### Kaggle Competition Submit

Genero las salidas y hago los submits a Kaggle

In [ ]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle

PARAM$kaggle$competencia <- "utn-2026-virtual-jr"
PARAM$kaggle$cortes <- seq(1800, 2400, by = 100)

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle", showWarnings= FALSE)

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marclo los primeros

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
  "  semilla=", PARAM$semilla_primigenia,
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  salida <- system(linea, intern=TRUE) # el submit a Kaggle
  Sys.sleep(30)
  cat(salida, "\n")
}

In [ ]:
# grabo los parametros
if( !require("yaml")) install.packages("yaml")
require("yaml")

write_yaml( PARAM, file="PARAM.yml")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")